# 03. Transformer Operational Stress Predictive Regressor
**Power Grid AI - Predictive Maintenance**

### Objectives:
- Train a Gradient Boosted Decision Tree regressor (`XGBoost` / `GradientBoostingRegressor`) to forecast transformer stress.
- Identify the most influential operating parameters driving asset thermal and mechanical degradation.
- Quantify accuracy via Root Mean Squared Error (RMSE), Mean Absolute Error (MAE), and $R^2$.


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load data and trained model
data_path = os.path.join('..', 'data', 'processed', 'cleaned_transformer_data.csv')
df = pd.read_csv(data_path)

model_path = os.path.join('..', 'models', 'xgboost_stress_model.pkl')
bundle = joblib.load(model_path)
model = bundle['model']
features = bundle['features']
metrics = bundle.get('metrics', {})

print(f"Loaded Stress Model. Saved Metrics: {metrics}")


### Feature Importance Analysis

In [ ]:
importances = model.feature_importances_
feat_imp = pd.Series(importances, index=features).sort_values(ascending=True)

plt.figure(figsize=(9, 6))
feat_imp.plot(kind='barh', color='teal')
plt.title('Stress Model Feature Importances', fontsize=13, fontweight='bold')
plt.xlabel('Relative Importance')
plt.tight_layout()
plt.show()


### Prediction vs Actual Stress Correlation

In [ ]:
y_true = df['stress_index']
y_pred = model.predict(df[features])

plt.figure(figsize=(8, 6))
plt.scatter(y_true, y_pred, alpha=0.5, color='darkblue', edgecolors='none')
plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=2, label='Perfect Fit')
plt.title('Actual Stress Index vs Model Predictions', fontsize=13, fontweight='bold')
plt.xlabel('Ground Truth Stress Index (0-100)')
plt.ylabel('Predicted Stress Index')
plt.legend()
plt.show()

print(f"Overall R² Score : {r2_score(y_true, y_pred):.4f}")
print(f"Overall MAE      : {mean_absolute_error(y_true, y_pred):.4f}")
print(f"Overall RMSE     : {np.sqrt(mean_squared_error(y_true, y_pred)):.4f}")
